# jitter_contour_bias_final

Deze tweede notebook werkt met jitter-afbeeldingen uit `/home/yentl/pytorch_gammanet/Images_jitter_final`.

Opbouw:
1. mean activation heatmaps voor `h1` per contour en jitterniveau;
2. baseline, C-bias en straight-bias met biaswaarde `0.25`;
3. heatmaps voor bias `0.25` bij jitter `0, 20, 40, 60`;
4. mask-based contour enrichment per jitterniveau en bias;
5. baseline-only contour enrichment vs jitter;
6. baseline vs matched bias contour enrichment vs jitter.

Deze notebook leest de channelclassificatie uit `outputs_bias_contour_final/csv/channel_classification`, dus run notebook 1 eerst.

In [ ]:
# ============================================================
# Imports, paths, settings
# ============================================================
import os, re, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from tqdm import tqdm

import torch
import torch.nn.functional as F
from torchvision import transforms

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

PROJECT_ROOT = Path("/home/yentl/pytorch_gammanet")
CHECKPOINT_PATH = PROJECT_ROOT / "checkpoint_epoch_40.pt"
INPUT_SIZE = (320, 320)
USE_ABS_ACTIVATION = True
TOP_ACTIVATION_PERCENTILE = 90

BOTTOM_UP_LAYERS = ["h0_exc", "h1_exc", "h2_exc", "h3_exc", "h4_exc"]
TOP_DOWN_LAYERS = ["td_h0_exc", "td_h1_exc", "td_h2_exc", "td_h3_exc"]
ALL_ANALYSIS_LAYERS = BOTTOM_UP_LAYERS + TOP_DOWN_LAYERS
CONTOURS = ["C", "straight"]

transform = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.ToTensor(),
])

IMAGE_DIR = PROJECT_ROOT / "Images_jitter_final"
MASK_DIR = PROJECT_ROOT / "contour_masks_jitter_final"
OUTPUT_DIR = PROJECT_ROOT / "outputs_jitter_contour_bias_final"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLASSIFICATION_DIR = PROJECT_ROOT / "outputs_bias_contour_final" / "csv" / "channel_classification"
C_CHANNELS_CSV = CLASSIFICATION_DIR / "C_channels_h1.csv"
STRAIGHT_CHANNELS_CSV = CLASSIFICATION_DIR / "straight_channels_h1.csv"

ANALYSIS_LAYER = "h1_exc"
BIAS_STRENGTH = 0.25
JITTER_LEVELS = [0, 10, 20, 30, 40, 50, 60]
SELECTED_OVERLAY_JITTERS = [0, 20, 40, 60]

print("IMAGE_DIR:", IMAGE_DIR)
print("MASK_DIR:", MASK_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CLASSIFICATION_DIR:", CLASSIFICATION_DIR)


## Belangrijk over maskers voor jitter

Als je `make_contour_masks.py` extern runt voor deze jittermap, pas dan aan:

```python
IMAGE_DIR = Path("/home/yentl/pytorch_gammanet/Images_jitter_final")
MASK_DIR = Path("/home/yentl/pytorch_gammanet/contour_masks_jitter_final")
n_points = 7
```

Omdat de contourpositie in de bestandsnaam zit, werkt dezelfde mask-logica ook voor alle jitterlevels. Deze notebook maakt ontbrekende maskers automatisch aan, dus extern runnen is optioneel.

In [ ]:
# ============================================================
# Model, parsing, activation and mask helpers
# ============================================================
def load_model():
    if str(PROJECT_ROOT) not in sys.path:
        sys.path.append(str(PROJECT_ROOT))
    from gammanet.models.vgg16_gammanet_v2 import VGG16GammaNetV2

    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    if "config" in checkpoint and "model" in checkpoint["config"]:
        model_config = checkpoint["config"]["model"]
    elif "model_config" in checkpoint:
        model_config = checkpoint["model_config"]
    else:
        raise KeyError("Could not find model config in checkpoint.")

    # Force 4 timesteps if you want to match the trained 4-timestep model behaviour.
    # Leave this commented if the checkpoint config already stores timesteps=4.
    # model_config["timesteps"] = 4

    model = VGG16GammaNetV2(model_config)
    state_dict = checkpoint.get("model_state_dict", checkpoint.get("state_dict", None))
    if state_dict is None:
        raise KeyError("Could not find model_state_dict or state_dict in checkpoint.")
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print("Missing keys:", len(missing))
    print("Unexpected keys:", len(unexpected))
    model.to(DEVICE)
    model.eval()
    print("Model timesteps:", model.timesteps)
    return model


def normalize_contour_label(label):
    lower = str(label).strip().lower()
    if lower in ["c", "ccontour", "c_contour"]:
        return "C"
    if lower in ["straight", "line", "straightline", "straight_line"]:
        return "straight"
    return label


def parse_image_filename(path):
    """
    Supports both filename styles:
    - C_BL_0_J000_000.png
    - C_high_BL_0_J000_000.png
    """
    parts = path.stem.split("_")
    if len(parts) == 5:
        contour, quadrant, position, jitter, stim_id = parts
        contrast = "NA"
    elif len(parts) >= 6:
        contour, contrast, quadrant, position, jitter, stim_id = parts[:6]
    else:
        return None
    contour_type = normalize_contour_label(contour)
    if contour_type not in CONTOURS:
        return None
    try:
        return {
            "filename": path.name,
            "path": str(path),
            "contour_type": contour_type,
            "contrast": contrast,
            "quadrant": quadrant,
            "position": int(position),
            "jitter": int(str(jitter).replace("J", "")),
            "stimulus_id": int(stim_id),
        }
    except Exception:
        return None


def load_image_tensor(path):
    pil_img = Image.open(path).convert("RGB")
    img_tensor = transform(pil_img).unsqueeze(0).to(DEVICE)
    return pil_img, img_tensor


def reset_and_forward(model, img_tensor):
    model.reset_hidden_states()
    with torch.no_grad():
        return model(img_tensor)


def get_state(model, layer_name):
    state = getattr(model, layer_name, None)
    if state is None:
        raise ValueError(f"{layer_name} is None. Run model first or check layer name.")
    return state


def normalize_for_plot(x, eps=1e-8):
    x = np.asarray(x)
    x = x - np.nanmin(x)
    return x / (np.nanmax(x) + eps)


def resize_map_to_image(fmap, pil_img):
    fmap_t = torch.tensor(fmap, dtype=torch.float32)[None, None]
    resized = F.interpolate(fmap_t, size=pil_img.size[::-1], mode="bilinear", align_corners=False)
    return resized[0, 0].numpy()


def all_channel_map(state, use_abs=True, channels=None):
    x = state.detach()
    if channels is not None:
        channels = [int(c) for c in channels]
        x = x[:, channels, :, :]
    if use_abs:
        x = x.abs()
    return x.mean(dim=1)[0].cpu().numpy()


def single_channel_map(state, channel):
    return state.detach().cpu()[0, int(channel)].numpy()

# ---------- Mask maker: straight now has 7 line elements ----------
N = 512
LINE_WIDTH = 18
STRAIGHT_N_POINTS = 7  # <- pas dit aan als je straight-masker meer/minder streepjes moet volgen
STRAIGHT_EDGE_MARGIN = 0.07


def bezier_curve_position(t, Ps):
    t = np.asarray(t)
    P0, P1, P2, P3 = Ps
    return (((1 - t) ** 3)[:, None] * P0 +
            (3 * ((1 - t) ** 2) * t)[:, None] * P1 +
            (3 * (1 - t) * (t ** 2))[:, None] * P2 +
            (t ** 3)[:, None] * P3)


def build_templates():
    Ps = np.array([[0.75, 0.9], [0.2, 0.9], [0.2, 0.1], [0.75, 0.1]])
    curve = bezier_curve_position(np.linspace(0, 1, 200), Ps * 0.5)
    bx_base = curve[:, 0][20:-20]
    by_base = curve[:, 1][20:-20]
    bx_base = bx_base - np.min(bx_base) + 0.07
    by_base = by_base - np.max(by_base) + 0.94
    templates = {}
    for shift_idx in range(4):
        bx = bx_base + 0.1 * shift_idx
        by = by_base.copy()
        templates[("C", shift_idx)] = (bx, by)
        x_center = np.mean(bx)
        templates[("bC", shift_idx)] = (2 * x_center - bx, by)
        bx_straight = np.full(STRAIGHT_N_POINTS, np.mean(bx))
        by_straight = np.linspace(np.min(by) + STRAIGHT_EDGE_MARGIN, np.max(by) - STRAIGHT_EDGE_MARGIN, STRAIGHT_N_POINTS)
        templates[("straight", shift_idx)] = (bx_straight, by_straight)
    return templates

TEMPLATES = build_templates()


def draw_template_mask(shape, position):
    bx, by = TEMPLATES[(shape, position)]
    points = list(zip(bx * N, by * N))
    mask = Image.new("L", (N, N), 0)
    draw = ImageDraw.Draw(mask)
    if shape == "straight":
        draw.line(points, fill=255, width=LINE_WIDTH)
    else:
        draw.line(points, fill=255, width=LINE_WIDTH, joint="curve")
    return mask


def apply_quadrant_transform(mask, original_shape, quadrant):
    effective_shape = original_shape
    if quadrant == "BL":
        pass
    elif quadrant == "BR":
        mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
        if original_shape == "C": effective_shape = "bC"
        elif original_shape == "bC": effective_shape = "C"
    elif quadrant == "TL":
        mask = mask.transpose(Image.FLIP_TOP_BOTTOM)
    elif quadrant == "TR":
        mask = mask.transpose(Image.FLIP_LEFT_RIGHT).transpose(Image.FLIP_TOP_BOTTOM)
        if original_shape == "C": effective_shape = "bC"
        elif original_shape == "bC": effective_shape = "C"
    else:
        raise ValueError(f"Unknown quadrant: {quadrant}")
    return mask, effective_shape


def make_mask_for_row(row, mask_dir):
    wanted_shape = row["contour_type"]
    quadrant = row["quadrant"]
    position = int(row["position"])
    base_shapes = ["straight"] if wanted_shape == "straight" else ["C", "bC"]
    candidate_masks = []
    for base_shape in base_shapes:
        mask = draw_template_mask(base_shape, position)
        mask, effective_shape = apply_quadrant_transform(mask, base_shape, quadrant)
        if effective_shape == wanted_shape:
            candidate_masks.append(mask)
    if not candidate_masks:
        return False
    final = Image.new("L", (N, N), 0)
    for mask in candidate_masks:
        final = Image.fromarray(np.maximum(np.asarray(final), np.asarray(mask)).astype(np.uint8))
    save_path = Path(mask_dir) / row["filename"]
    final.save(save_path)
    return True


def ensure_masks(df, mask_dir):
    mask_dir = Path(mask_dir)
    mask_dir.mkdir(parents=True, exist_ok=True)
    n_made, n_exists = 0, 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Checking/making masks"):
        if (mask_dir / row["filename"]).exists():
            n_exists += 1
        else:
            n_made += int(make_mask_for_row(row, mask_dir))
    print(f"Masks already present: {n_exists}; masks created: {n_made}; mask dir: {mask_dir}")


def load_true_contour_mask(row, target_size, mask_dir):
    mask_path = Path(mask_dir) / row["filename"]
    if not mask_path.exists():
        make_mask_for_row(row, mask_dir)
    mask = Image.open(mask_path).convert("L")
    mask = mask.resize(target_size, resample=Image.NEAREST)
    return np.asarray(mask) > 0


def compute_mask_metrics_from_activation(act, contour_mask):
    act = np.asarray(act)
    act = np.abs(act) if USE_ABS_ACTIVATION else act.copy()
    act = np.nan_to_num(act, nan=0.0, posinf=0.0, neginf=0.0)
    act = act - act.min() + 1e-8
    background_mask = ~contour_mask
    contour_values = act[contour_mask]
    background_values = act[background_mask]
    contour_mean = contour_values.mean()
    background_mean = background_values.mean()
    contour_sum = contour_values.sum()
    total_sum = act.sum()
    contour_area_pct = 100 * contour_mask.mean()
    activation_on_contour_pct = 100 * contour_sum / (total_sum + 1e-8)
    contour_enrichment = activation_on_contour_pct / (contour_area_pct + 1e-8)
    act_threshold = np.percentile(act, TOP_ACTIVATION_PERCENTILE)
    top_activation_mask = act >= act_threshold
    intersection = np.logical_and(contour_mask, top_activation_mask).sum()
    dice_top_activation = 2 * intersection / (contour_mask.sum() + top_activation_mask.sum() + 1e-8)
    return {
        "contour_mean_activation": contour_mean,
        "background_mean_activation": background_mean,
        "contour_preference_ratio": contour_mean / (background_mean + 1e-8),
        "activation_on_contour_pct": activation_on_contour_pct,
        "contour_area_pct": contour_area_pct,
        "contour_enrichment": contour_enrichment,
        "dice_top_activation": dice_top_activation,
        "top_activation_percentile": TOP_ACTIVATION_PERCENTILE,
    }


def plot_state_overlay(state, pil_img, title, save_path=None, channels=None, alpha=0.55):
    fmap = all_channel_map(state, use_abs=USE_ABS_ACTIVATION, channels=channels)
    fmap_resized = resize_map_to_image(fmap, pil_img)
    fmap_norm = normalize_for_plot(fmap_resized)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(pil_img); axes[0].set_title("Stimulus"); axes[0].axis("off")
    im = axes[1].imshow(fmap_norm, cmap="inferno"); axes[1].set_title(title); axes[1].axis("off")
    fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
    axes[2].imshow(pil_img); axes[2].imshow(fmap_norm, cmap="inferno", alpha=alpha)
    axes[2].set_title("Overlay"); axes[2].axis("off")
    plt.tight_layout()
    if save_path is not None:
        save_path = Path(save_path); save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print("Saved:", save_path)
    plt.show()
    plt.close(fig)

# ============================================================
# Top-down bias hook on td_h1_exc channels
# ============================================================
class TDH1Bias:
    """Adds a non-invasive top-down bias after td_fgru_1, i.e. to td_h1 channels."""
    def __init__(self, model, channels, strength=0.25, mode="add_mean_abs"):
        self.model = model
        self.channels = [int(c) for c in channels]
        self.strength = float(strength)
        self.mode = mode
        self.handle = None

    def _hook(self, module, inputs, output):
        if not isinstance(output, tuple):
            return output
        exc = output[0]
        if exc is None or len(self.channels) == 0:
            return output
        exc = exc.clone()
        idx = torch.tensor(self.channels, device=exc.device, dtype=torch.long)
        if self.mode == "add_mean_abs":
            scale = exc.detach().abs().mean(dim=(2, 3), keepdim=True) + 1e-8
            exc[:, idx, :, :] = exc[:, idx, :, :] + self.strength * scale[:, idx, :, :]
        elif self.mode == "multiply":
            exc[:, idx, :, :] = exc[:, idx, :, :] * (1.0 + self.strength)
        else:
            raise ValueError("Unknown mode")
        return (exc,) + output[1:]

    def __enter__(self):
        self.handle = self.model.td_fgru_1.register_forward_hook(self._hook)
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if self.handle is not None:
            self.handle.remove()
        return False


def forward_condition(model, img_tensor, condition, C_CHANNELS, STRAIGHT_CHANNELS):
    if condition == "baseline":
        return reset_and_forward(model, img_tensor)
    if condition == "C_bias":
        with TDH1Bias(model, C_CHANNELS, strength=BIAS_STRENGTH):
            return reset_and_forward(model, img_tensor)
    if condition == "straight_bias":
        with TDH1Bias(model, STRAIGHT_CHANNELS, strength=BIAS_STRENGTH):
            return reset_and_forward(model, img_tensor)
    raise ValueError(condition)


def matched_condition_for_contour(contour):
    return "C_bias" if contour == "C" else "straight_bias"


In [ ]:
# ============================================================
# Load model, channel classes, and jitter image table
# ============================================================
model = load_model()

if not C_CHANNELS_CSV.exists() or not STRAIGHT_CHANNELS_CSV.exists():
    raise FileNotFoundError(
        "Channel classification files not found. Run bias_contour_final.ipynb first, "
        f"or update CLASSIFICATION_DIR: {CLASSIFICATION_DIR}"
    )
C_CHANNELS = pd.read_csv(C_CHANNELS_CSV)["channel"].astype(int).tolist()
STRAIGHT_CHANNELS = pd.read_csv(STRAIGHT_CHANNELS_CSV)["channel"].astype(int).tolist()
print(f"Loaded {len(C_CHANNELS)} C channels and {len(STRAIGHT_CHANNELS)} straight channels")

rows = []
for path in sorted(IMAGE_DIR.glob("*.png")):
    info = parse_image_filename(path)
    if info is not None and info["jitter"] in JITTER_LEVELS:
        rows.append(info)

jitter_df = pd.DataFrame(rows)
if jitter_df.empty:
    raise RuntimeError(f"No C/straight jitter images found in {IMAGE_DIR}")
jitter_df = jitter_df.sort_values(["contour_type", "jitter", "position", "quadrant", "stimulus_id"]).reset_index(drop=True)

print(jitter_df.groupby(["contour_type", "jitter"]).size())
(OUTPUT_DIR / "csv").mkdir(parents=True, exist_ok=True)
jitter_df.to_csv(OUTPUT_DIR / "csv" / "jitter_image_table.csv", index=False)
display(jitter_df.head())


In [ ]:
# ============================================================
# 1. Mean h1 activation heatmaps per jitter level, baseline
# ============================================================
HEATMAP_DIR = OUTPUT_DIR / "plots" / "01_h1_mean_heatmaps_by_jitter_baseline"
HEATMAP_DIR.mkdir(parents=True, exist_ok=True)

mean_h1_by_contour_jitter = {c: {} for c in CONTOURS}

for contour in CONTOURS:
    for jitter in JITTER_LEVELS:
        sub = jitter_df[(jitter_df["contour_type"] == contour) & (jitter_df["jitter"] == jitter)]
        if sub.empty:
            print(f"Skipping {contour} J{jitter:03d}: no images")
            continue
        state_sum = None
        for _, row in tqdm(sub.iterrows(), total=len(sub), desc=f"Baseline {contour} J{jitter:03d}"):
            pil_img, img_tensor = load_image_tensor(row["path"])
            reset_and_forward(model, img_tensor)
            state = get_state(model, ANALYSIS_LAYER).detach().cpu()
            state_sum = state.clone() if state_sum is None else state_sum + state
        mean_state = state_sum / len(sub)
        mean_h1_by_contour_jitter[contour][jitter] = mean_state
        example_row = sub.iloc[0]
        pil_img, _ = load_image_tensor(example_row["path"])
        plot_state_overlay(
            mean_state,
            pil_img,
            title=f"Baseline | {contour} | J{jitter:03d} | {ANALYSIS_LAYER}",
            save_path=HEATMAP_DIR / f"baseline_mean_h1_{contour}_J{jitter:03d}.png",
        )


In [ ]:
# ============================================================
# 2 + 4. Run baseline, C-bias and straight-bias; quantify contour enrichment
# ============================================================
ensure_masks(jitter_df, MASK_DIR)

CONDITIONS = ["baseline", "C_bias", "straight_bias"]
metric_rows = []

for _, row in tqdm(jitter_df.iterrows(), total=len(jitter_df), desc="Quantify jitter conditions"):
    pil_img, img_tensor = load_image_tensor(row["path"])
    contour_mask = load_true_contour_mask(row, target_size=pil_img.size, mask_dir=MASK_DIR)

    for condition in CONDITIONS:
        forward_condition(model, img_tensor, condition, C_CHANNELS, STRAIGHT_CHANNELS)
        state = get_state(model, ANALYSIS_LAYER).detach().cpu()

        # all-channel h1 activation for the main enrichment metric
        fmap = all_channel_map(state, use_abs=USE_ABS_ACTIVATION)
        fmap_resized = resize_map_to_image(fmap, pil_img)
        metrics = compute_mask_metrics_from_activation(fmap_resized, contour_mask)

        metric_rows.append({
            **{k: row[k] for k in ["filename", "contour_type", "contrast", "quadrant", "position", "jitter", "stimulus_id"]},
            "condition": condition,
            "bias_strength": 0.0 if condition == "baseline" else BIAS_STRENGTH,
            "layer": ANALYSIS_LAYER,
            "channel_group": "all_channels",
            "n_channels": int(state.shape[1]),
            **metrics,
        })

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(OUTPUT_DIR / "csv" / "jitter_h1_contour_enrichment_per_image_all_conditions.csv", index=False)

summary_df = (
    metrics_df
    .groupby(["contour_type", "jitter", "condition", "bias_strength", "layer", "channel_group"], as_index=False)
    .agg(
        n_images=("filename", "nunique"),
        mean_contour_enrichment=("contour_enrichment", "mean"),
        sem_contour_enrichment=("contour_enrichment", "sem"),
        mean_contour_preference_ratio=("contour_preference_ratio", "mean"),
        sem_contour_preference_ratio=("contour_preference_ratio", "sem"),
        mean_activation_on_contour_pct=("activation_on_contour_pct", "mean"),
        sem_activation_on_contour_pct=("activation_on_contour_pct", "sem"),
        mean_dice_top_activation=("dice_top_activation", "mean"),
        sem_dice_top_activation=("dice_top_activation", "sem"),
    )
)
summary_df.to_csv(OUTPUT_DIR / "csv" / "jitter_h1_contour_enrichment_summary.csv", index=False)
display(summary_df)


In [ ]:
# ============================================================
# 3. Heatmaps for matched bias 0.25 at J000, J020, J040, J060
# ============================================================
BIAS_HEATMAP_DIR = OUTPUT_DIR / "plots" / "03_matched_bias_heatmaps_selected_jitters"
BIAS_HEATMAP_DIR.mkdir(parents=True, exist_ok=True)

for contour in CONTOURS:
    condition = matched_condition_for_contour(contour)
    for jitter in SELECTED_OVERLAY_JITTERS:
        sub = jitter_df[(jitter_df["contour_type"] == contour) & (jitter_df["jitter"] == jitter)]
        if sub.empty:
            print(f"Skipping {contour} J{jitter:03d}: no images")
            continue
        state_sum = None
        for _, row in tqdm(sub.iterrows(), total=len(sub), desc=f"{condition} {contour} J{jitter:03d}"):
            pil_img, img_tensor = load_image_tensor(row["path"])
            forward_condition(model, img_tensor, condition, C_CHANNELS, STRAIGHT_CHANNELS)
            state = get_state(model, ANALYSIS_LAYER).detach().cpu()
            state_sum = state.clone() if state_sum is None else state_sum + state
        mean_state = state_sum / len(sub)
        example_row = sub.iloc[0]
        pil_img, _ = load_image_tensor(example_row["path"])
        plot_state_overlay(
            mean_state,
            pil_img,
            title=f"{condition} {BIAS_STRENGTH} | {contour} | J{jitter:03d} | {ANALYSIS_LAYER}",
            save_path=BIAS_HEATMAP_DIR / f"{condition}_mean_h1_{contour}_J{jitter:03d}.png",
        )


In [ ]:
# ============================================================
# 5. Baseline-only contour enrichment vs jitter
# ============================================================
PLOT_DIR = OUTPUT_DIR / "plots" / "05_06_enrichment_vs_jitter"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

for contour in CONTOURS:
    sub = summary_df[(summary_df["contour_type"] == contour) & (summary_df["condition"] == "baseline")].sort_values("jitter")
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.errorbar(sub["jitter"], sub["mean_contour_enrichment"], yerr=sub["sem_contour_enrichment"], marker="o", linewidth=2, capsize=4, label="baseline")
    ax.set_xlabel("Jitter")
    ax.set_ylabel("Contour enrichment")
    ax.set_title(f"Baseline contour enrichment vs jitter | {contour}")
    ax.set_xticks(JITTER_LEVELS)
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    save_path = PLOT_DIR / f"baseline_enrichment_vs_jitter_{contour}.png"
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    print("Saved:", save_path)
    plt.show(); plt.close(fig)


In [ ]:
# ============================================================
# 6. Baseline vs matched bias 0.25 contour enrichment vs jitter
# ============================================================
for contour in CONTOURS:
    matched = matched_condition_for_contour(contour)
    sub = summary_df[(summary_df["contour_type"] == contour) & (summary_df["condition"].isin(["baseline", matched]))].sort_values("jitter")
    fig, ax = plt.subplots(figsize=(7, 5))
    for condition in ["baseline", matched]:
        csub = sub[sub["condition"] == condition].sort_values("jitter")
        label = "baseline" if condition == "baseline" else f"{condition} ({BIAS_STRENGTH})"
        ax.errorbar(csub["jitter"], csub["mean_contour_enrichment"], yerr=csub["sem_contour_enrichment"], marker="o", linewidth=2, capsize=4, label=label)
    ax.set_xlabel("Jitter")
    ax.set_ylabel("Contour enrichment")
    ax.set_title(f"Contour enrichment vs jitter | {contour}")
    ax.set_xticks(JITTER_LEVELS)
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    save_path = PLOT_DIR / f"baseline_vs_matched_bias_enrichment_vs_jitter_{contour}.png"
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    print("Saved:", save_path)
    plt.show(); plt.close(fig)
